# CryptoQuant HDF5 数据查看

这个 notebook 用于分别查看 `data/crypto_quant.h5` 中的各个表。

使用顺序：先运行路径和工具函数单元格，再运行总览单元格或下面对应表的查看单元格。默认只展示前若干行，避免一次性输出整张表。

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import tables

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

# 兼容从仓库根目录或 data/ 目录启动 Jupyter。
_candidates = [
    Path('data/crypto_quant.h5'),
    Path('crypto_quant.h5'),
    Path.cwd().parent / 'data' / 'crypto_quant.h5',
]
H5_PATH = next((path.resolve() for path in _candidates if path.is_file()), None)
if H5_PATH is None:
    raise FileNotFoundError(
        '找不到 crypto_quant.h5，请确认 notebook 位于 data/ 目录，'
        '或从仓库根目录启动，并先运行 data/update_crypto_quant.py'
    )

print(f'HDF5 文件: {H5_PATH}')
print(f'文件大小: {H5_PATH.stat().st_size / 1024**2:.2f} MB')

HDF5 文件: /Users/dmiwu/work/PythonProject/cryptoFactorAnalyze/data/crypto_quant.h5
文件大小: 183.16 MB


## 工具函数

- `list_tables()`：列出所有表、行数、字段和日期范围。
- `load_table(name)`：加载指定表的完整 DataFrame。
- `preview_table(name)`：按行数和日期范围查看指定表。

表名可以写成 `cmc100_daily` 或 `/cmc100_daily`。

In [ ]:
def _table_name(name: str) -> str:
    return name.lstrip('/')


def _stored_columns(storer) -> list[str]:
    """从 HDF5 storer 元数据读取 DataFrame 的真实字段名。"""
    for axis, columns in getattr(storer.attrs, 'non_index_axes', []):
        if axis == 1:
            return list(columns)
    return [column for column in storer.table.colnames if not column.startswith('values_block_')]


def _decode_string(value):
    if isinstance(value, (bytes, np.bytes_)):
        return value.decode('utf-8', errors='replace').rstrip('\x00')
    return value


def _datetime_unit(dtype) -> str:
    text = str(dtype)
    return text.split('[', 1)[1].split(']', 1)[0] if '[' in text else 'ns'


def _read_hdf_table(name: str) -> pd.DataFrame:
    """通过 PyTables 读取，兼容 pandas 1.5 对不同 datetime unit 的断言问题。"""
    name = _table_name(name)
    with pd.HDFStore(H5_PATH, mode='r') as store:
        storer = store.get_storer(name)
        logical_columns = list(storer.attrs.non_index_axes[0][1])
        axes = list(storer.values_axes)
        index_name = next((item[1] for item in getattr(storer.attrs, 'index_cols', []) if item[0] == 0), 'index')

    column_data = {}
    with tables.open_file(H5_PATH, mode='r') as h5:
        node = h5.get_node(f'/{name}/table')
        index_values = node.col('index')
        for axis in axes:
            raw = node.col(axis.name)
            names = list(axis.values)
            if len(names) == 1 and getattr(raw, 'ndim', 1) == 2:
                raw = raw[:, 0]
            if str(axis.kind).startswith('datetime64') or str(axis.dtype).startswith('datetime64['):
                unit = _datetime_unit(axis.dtype)
                if getattr(raw, 'ndim', 1) == 1:
                    converted = pd.to_datetime(raw, unit=unit, errors='coerce', utc=True)
                    if getattr(axis, 'tz', None) is None:
                        converted = converted.tz_localize(None)
                    column_data[names[0]] = converted
                else:
                    for index, column in enumerate(names):
                        converted = pd.to_datetime(raw[:, index], unit=unit, errors='coerce', utc=True)
                        if getattr(axis, 'tz', None) is None:
                            converted = converted.tz_localize(None)
                        column_data[column] = converted
            elif len(names) == 1:
                values = raw
                if axis.kind == 'string':
                    values = [_decode_string(value) for value in values]
                column_data[names[0]] = values
            else:
                for index, column in enumerate(names):
                    column_data[column] = raw[:, index]
    result = pd.DataFrame({column: column_data[column] for column in logical_columns}, index=index_values)
    result.index.name = index_name
    return result


def list_tables() -> pd.DataFrame:
    descriptors = []
    with pd.HDFStore(H5_PATH, mode='r') as store:
        for key in store.keys():
            name = _table_name(key)
            storer = store.get_storer(name)
            descriptors.append((name, storer.nrows, _stored_columns(storer)))
    rows = []
    for name, row_count, columns in descriptors:
        date_range = ''
        date_column = next((column for column in ('date', 'decision_date', 'funding_time') if column in columns), None)
        if date_column is not None:
            dates = pd.to_datetime(_read_hdf_table(name)[date_column], errors='coerce', utc=True).dropna()
            if not dates.empty:
                date_range = f'{dates.min()} ~ {dates.max()}'
        rows.append({'table': name, 'rows': row_count, 'columns': ', '.join(columns), 'date_range': date_range})
    return pd.DataFrame(rows).sort_values('table').reset_index(drop=True)


def load_table(name: str) -> pd.DataFrame:
    """加载指定表的完整内容；大表建议优先使用 preview_table。"""
    return _read_hdf_table(name)


def preview_table(
    name: str,
    rows: int = 10,
    start_date: str | None = None,
    end_date: str | None = None,
) -> pd.DataFrame:
    """查看单张表，可选按 date/decision_date/funding_time 过滤。"""
    frame = load_table(name)
    date_column = next((column for column in ('date', 'decision_date', 'funding_time') if column in frame.columns), None)
    if date_column is not None and (start_date or end_date):
        values = pd.to_datetime(frame[date_column], errors='coerce', utc=True)
        if start_date:
            values_start = pd.Timestamp(start_date, tz='UTC')
            frame = frame.loc[values >= values_start]
            values = values.loc[frame.index]
        if end_date:
            values_end = pd.Timestamp(end_date, tz='UTC') + pd.Timedelta(days=1)
            frame = frame.loc[values < values_end]
    return frame.head(rows).reset_index(drop=True)


def show_table(name: str, rows: int = 10, start_date: str | None = None, end_date: str | None = None) -> None:
    print(f'/{_table_name(name)}')
    display(preview_table(name, rows=rows, start_date=start_date, end_date=end_date))

## 1. 所有表总览

In [3]:
tables_overview = list_tables()
display(tables_overview)

AssertionError: (updated_at_utc    datetime64[ns]
dtype: object, dtype('<M8[us]'))

## 2. 分别查看每个表

下面每个单元格只展示前 10 行。可以修改 `rows`，也可以增加 `start_date` / `end_date`，例如：

```python
show_table('research_panel_daily', rows=20, start_date='2024-01-01', end_date='2024-01-10')
```

In [ ]:
show_table('cmc100_daily')
show_table('cmc100_constituents')

In [ ]:
show_table('futures_contracts')
show_table('klines_daily')

In [ ]:
show_table('funding_events')
show_table('universe_monthly')

In [ ]:
show_table('research_panel_daily')
show_table('_metadata')

## 3. 常用查询示例

### 查看某天的 Top50

```python
universe = load_table('universe_monthly')
universe[universe['effective_date'].eq(pd.Timestamp('2024-01-02'))]
```

### 查看某个交易对的 K 线

```python
klines = load_table('klines_daily')
klines[klines['symbol'].eq('BTCUSDT')].tail(20)
```

### 查看研究面板的完整性标记

```python
panel = load_table('research_panel_daily')
panel[['date', 'binance_symbol', 'has_complete_kline', 'has_complete_funding']].head()
```